# 12 Publication-Quality Figures and Tables

Create paper-ready figures and tables from the report artifacts generated by earlier notebooks. Outputs are written as vector PDFs/SVGs and high-resolution PNGs under reports/publication/.


In [1]:
from pathlib import Path
import json

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_curve, average_precision_score

mpl.use('Agg')

In [2]:
REPO_ROOT = Path('..').resolve()
REPORT_DIR = REPO_ROOT / 'reports'
PUBLICATION_DIR = REPORT_DIR / 'publication'
FIGURE_DIR = PUBLICATION_DIR / 'figures'
TABLE_DIR = PUBLICATION_DIR / 'tables'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_SUMMARY_PATH = REPORT_DIR / 'manifests' / 'turning_split_summary.csv'
BASELINE_METRICS_PATH = REPORT_DIR / 'tables' / 'metrics_turning_baselines.csv'
BASELINE_METRICS_CI_PATH = REPORT_DIR / 'tables' / 'metrics_turning_baselines_with_ci.csv'
BASELINE_SCORES_PATH = REPORT_DIR / 'baselines' / 'baseline_scores_turning.csv'
BASELINE_THRESHOLDS_PATH = REPORT_DIR / 'thresholds' / 'baseline_thresholds_turning.json'
AE_SCORES_PATH = REPORT_DIR / 'scores' / 'ae_scores_turning.csv'
AE_THRESHOLDS_PATH = REPORT_DIR / 'thresholds' / 'ae_thresholds_turning.json'

METHOD_LABELS = {
    'one_class_svm_image_features': 'One-class SVM',
    'isolation_forest_image_features': 'Isolation forest',
    'pca_image_reconstruction': 'PCA reconstruction',
}
METRIC_LABELS = {
    'pr_auc': 'PR-AUC',
    'f1': 'F1-score',
    'precision': 'Precision',
    'recall': 'Recall',
}

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'legend.fontsize': 7,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})


In [3]:
def save_figure(fig, stem: str) -> None:
    for ext in ['pdf', 'svg', 'png']:
        fig.savefig(FIGURE_DIR / f'{stem}.{ext}', bbox_inches='tight')
    plt.close(fig)

def interpolated_precision_recall(y_true, scores):
    precision, recall, _ = precision_recall_curve(y_true, scores)
    order = np.argsort(recall)
    recall = recall[order]
    precision = precision[order]
    recall_unique = np.unique(recall)
    precision_unique = np.array([precision[recall == value].max() for value in recall_unique])
    precision_envelope = np.maximum.accumulate(precision_unique[::-1])[::-1]
    return recall_unique, precision_envelope

split_summary = pd.read_csv(SPLIT_SUMMARY_PATH)
baseline_metrics = pd.read_csv(BASELINE_METRICS_CI_PATH if BASELINE_METRICS_CI_PATH.exists() else BASELINE_METRICS_PATH)
baseline_scores = pd.read_csv(BASELINE_SCORES_PATH)
with BASELINE_THRESHOLDS_PATH.open() as f:
    baseline_thresholds = json.load(f)

baseline_metrics['method_label'] = baseline_metrics['method'].map(METHOD_LABELS).fillna(baseline_metrics['method'])
baseline_scores['method_label'] = baseline_scores['method'].map(METHOD_LABELS).fillna(baseline_scores['method'])


## Dataset Split Figure and Table


In [4]:
split_pivot = split_summary.pivot_table(index='split', columns='label', values='n', fill_value=0)
split_pivot = split_pivot.reindex(['train', 'validation', 'test']).fillna(0)
split_pivot = split_pivot.rename(columns={'no_chatter': 'Nominal', 'chatter': 'Chatter'})

fig, ax = plt.subplots(figsize=(3.35, 2.1))
bottom = np.zeros(len(split_pivot))
colors = {'Nominal': '#4C78A8', 'Chatter': '#F58518'}
for label in ['Nominal', 'Chatter']:
    values = split_pivot[label].to_numpy() if label in split_pivot else np.zeros(len(split_pivot))
    ax.bar(split_pivot.index, values, bottom=bottom, label=label, color=colors[label], width=0.62)
    for x, y0, value in zip(range(len(values)), bottom, values):
        if value > 0:
            ax.text(x, y0 + value / 2, f'{int(value)}', ha='center', va='center', color='white', fontsize=7)
    bottom += values
ax.set_ylabel('Number of windows')
ax.set_xlabel('Split')
ax.legend(frameon=False, loc='upper right')
ax.set_title('Frozen turning-dataset split')
ax.grid(axis='y', color='0.9', linewidth=0.8)
save_figure(fig, 'turning_split_counts')

split_pivot.to_csv(TABLE_DIR / 'turning_split_counts_publication.csv')
split_pivot.to_latex(TABLE_DIR / 'turning_split_counts_publication.tex')
split_pivot


label,Chatter,Nominal
split,,
train,0.0,472.0
validation,34.0,70.0
test,27.0,48.0


## Baseline Metric Figure and Table


In [5]:

metric_order = ['pr_auc', 'f1', 'precision', 'recall']

plot_table = (
    baseline_metrics
    .set_index('method_label')
    .loc[[METHOD_LABELS[m] for m in METHOD_LABELS], metric_order]
)

fig, ax = plt.subplots(figsize=(6.8, 2.45))

x = np.arange(len(plot_table.index))
bar_width = 0.18

palette = [
    '#4C78A8',
    '#F58518',
    '#54A24B',
    '#B279A2',
]

for i, metric in enumerate(metric_order):

    values = plot_table[metric].to_numpy()
    offsets = x + (i - 1.5) * bar_width

    ax.bar(
        offsets,
        values,
        width=bar_width,
        label=METRIC_LABELS[metric],
        color=palette[i],
    )

ax.set_xticks(x)
ax.set_xticklabels(plot_table.index)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')

ax.set_title(
    'Baseline anomaly-detection performance on held-out test split'
)

ax.legend(
    frameon=False,
    ncol=4,
    loc='lower center',
    bbox_to_anchor=(0.5, 1.02),
)

ax.grid(
    axis='y',
    color='0.9',
    linewidth=0.8,
)

save_figure(
    fig,
    'baseline_metric_panel',
)

# ============================================================
# Turning dataset comparison table
# ============================================================

ae_metrics = pd.read_csv(
    REPORT_DIR / 'tables' / 'metrics_turning_ae.csv'
)

# CNN-AE rows
score_labels = {
    'global_mse': 'Global MSE',
    'global_mae': 'Global MAE',
    'sve_max': 'SVE maximum',
    'sve_topk': 'SVE top-k',
}

comparison_rows = []

for score, label in score_labels.items():

    row = ae_metrics[
        ae_metrics['score'] == score
    ].iloc[0]

    comparison_rows.append({
        'Method': 'CNN-AE',
        'Score': label,
        'PR-AUC': row['pr_auc'],
        'F1': row['f1'],
        'Precision': row['precision'],
        'Recall': row['recall'],
    })

# Baselines
baseline_score_labels = {
    'One-class SVM': 'Anomaly score',
    'Isolation forest': 'Anomaly score',
    'PCA reconstruction': 'MSE',
}

for _, row in baseline_metrics.iterrows():

    comparison_rows.append({
        'Method': row['method_label'],
        'Score': baseline_score_labels[row['method_label']],
        'PR-AUC': row['pr_auc'],
        'F1': row['f1'],
        'Precision': row['precision'],
        'Recall': row['recall'],
    })

publication_metrics = pd.DataFrame(comparison_rows)

publication_metrics.to_csv(
    TABLE_DIR / 'turning_method_comparison.csv',
    index=False,
)

publication_metrics.to_latex(
    TABLE_DIR / 'turning_method_comparison.tex',
    index=False,
    float_format='%.3f',
)

publication_metrics

,Method,Score,PR-AUC,F1,Precision,Recall
0,CNN-AE,Global MSE,0.974098,0.915254,0.843750,1.000000
1,CNN-AE,Global MAE,0.988484,0.912281,0.866667,0.962963
2,CNN-AE,SVE maximum,0.956133,0.931034,0.870968,1.000000
3,CNN-AE,SVE top-k,0.962457,0.931034,0.870968,1.000000
4,Isolation forest,Anomaly score,0.997306,0.981818,0.964286,1.000000
5,One-class SVM,Anomaly score,0.997306,0.961538,1.000000,0.925926
6,PCA reconstruction,MSE,0.965310,0.947368,0.900000,1.000000


## Precision-Recall Curves


In [6]:
fig, ax = plt.subplots(figsize=(3.35, 2.65))
for method, group in baseline_scores[baseline_scores['split'] == 'test'].groupby('method'):
    y_true = group['target'].to_numpy()
    scores = group['score_value'].to_numpy()
    recall, precision = interpolated_precision_recall(y_true, scores)
    ap = average_precision_score(y_true, scores)
    ax.step(recall, precision, where='post', linewidth=1.8, label=f'{METHOD_LABELS.get(method, method)} ({ap:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.set_title('Interpolated precision-recall curves')
ax.grid(color='0.9', linewidth=0.8)
ax.legend(frameon=False, loc='lower left')
save_figure(fig, 'baseline_precision_recall_curves')


## Confusion Matrices


In [7]:
methods = list(METHOD_LABELS)
fig, axes = plt.subplots(1, len(methods), figsize=(6.8, 2.25), constrained_layout=True)
if len(methods) == 1:
    axes = [axes]
for ax, method in zip(axes, methods):
    group = baseline_scores[(baseline_scores['split'] == 'test') & (baseline_scores['method'] == method)]
    threshold = baseline_thresholds[method]['threshold']
    y_true = group['target'].to_numpy()
    y_pred = (group['score_value'].to_numpy() >= threshold).astype(int)
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    ax.imshow(matrix, cmap='Blues', vmin=0)
    for (row, col), value in np.ndenumerate(matrix):
        ax.text(col, row, str(value), ha='center', va='center', color='black', fontsize=9)
    ax.set_xticks([0, 1], labels=['Nominal', 'Chatter'], rotation=35, ha='right')
    ax.set_yticks([0, 1], labels=['Nominal', 'Chatter'])
    ax.set_xlabel('Predicted')
    ax.set_title(METHOD_LABELS.get(method, method))
axes[0].set_ylabel('True')
save_figure(fig, 'baseline_confusion_matrices')


## CNN-AE Architecture Figure and Table

These publication artifacts document the CNN autoencoder architecture used in notebook 04. They do not require TensorFlow and can be regenerated from the model specification in the notebook.


In [8]:
ae_layers = pd.DataFrame([
    {'stage': 'Input', 'layer': 'Input image', 'type': 'Input', 'kernel': '', 'stride_pool': '', 'padding': '', 'activation': '', 'output_shape': '100 x 150 x 3', 'params': 0},
    {'stage': 'Encoder', 'layer': 'Conv2D-4', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '100 x 150 x 4', 'params': 112},
    {'stage': 'Encoder', 'layer': 'MaxPool', 'type': 'MaxPooling2D', 'kernel': '', 'stride_pool': '2 x 2', 'padding': 'same', 'activation': '', 'output_shape': '50 x 75 x 4', 'params': 0},
    {'stage': 'Encoder', 'layer': 'Conv2D-8', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '50 x 75 x 8', 'params': 296},
    {'stage': 'Encoder', 'layer': 'MaxPool', 'type': 'MaxPooling2D', 'kernel': '', 'stride_pool': '2 x 3', 'padding': 'same', 'activation': '', 'output_shape': '25 x 25 x 8', 'params': 0},
    {'stage': 'Encoder', 'layer': 'Conv2D-12', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '25 x 25 x 12', 'params': 876},
    {'stage': 'Bottleneck', 'layer': 'Flatten', 'type': 'Flatten', 'kernel': '', 'stride_pool': '', 'padding': '', 'activation': '', 'output_shape': '7500', 'params': 0},
    {'stage': 'Bottleneck', 'layer': 'Dense-16', 'type': 'Dense', 'kernel': '', 'stride_pool': '', 'padding': '', 'activation': 'ReLU', 'output_shape': '16', 'params': 120016},
    {'stage': 'Decoder', 'layer': 'Dense-7500', 'type': 'Dense', 'kernel': '', 'stride_pool': '', 'padding': '', 'activation': 'ReLU', 'output_shape': '7500', 'params': 127500},
    {'stage': 'Decoder', 'layer': 'Reshape', 'type': 'Reshape', 'kernel': '', 'stride_pool': '', 'padding': '', 'activation': '', 'output_shape': '25 x 25 x 12', 'params': 0},
    {'stage': 'Decoder', 'layer': 'Conv2D-12', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '25 x 25 x 12', 'params': 1308},
    {'stage': 'Decoder', 'layer': 'UpSampling', 'type': 'UpSampling2D', 'kernel': '', 'stride_pool': '2 x 3', 'padding': '', 'activation': '', 'output_shape': '50 x 75 x 12', 'params': 0},
    {'stage': 'Decoder', 'layer': 'Conv2D-8', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '50 x 75 x 8', 'params': 872},
    {'stage': 'Decoder', 'layer': 'UpSampling', 'type': 'UpSampling2D', 'kernel': '', 'stride_pool': '2 x 2', 'padding': '', 'activation': '', 'output_shape': '100 x 150 x 8', 'params': 0},
    {'stage': 'Decoder', 'layer': 'Conv2D-4', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '100 x 150 x 4', 'params': 292},
    {'stage': 'Output', 'layer': 'Conv2D-3', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'Sigmoid', 'output_shape': '100 x 150 x 3', 'params': 111},
])
ae_layers.to_csv(TABLE_DIR / 'cnn_ae_architecture_publication.csv', index=False)
ae_layers.to_latex(TABLE_DIR / 'cnn_ae_architecture_publication.tex', index=False)
print('Total parameters:', int(ae_layers['params'].sum()))
ae_layers


Total parameters: 251383


,stage,layer,type,kernel,stride_pool,padding,activation,output_shape,params
0,Input,Input image,Input,,,,,100 x 150 x 3,0
1,Encoder,Conv2D-4,Conv2D,3 x 3,1 x 1,same,ReLU,100 x 150 x 4,112
2,Encoder,MaxPool,MaxPooling2D,,2 x 2,same,,50 x 75 x 4,0
3,Encoder,Conv2D-8,Conv2D,3 x 3,1 x 1,same,ReLU,50 x 75 x 8,296
4,Encoder,MaxPool,MaxPooling2D,,2 x 3,same,,25 x 25 x 8,0
5,Encoder,Conv2D-12,Conv2D,3 x 3,1 x 1,same,ReLU,25 x 25 x 12,876
6,Bottleneck,Flatten,Flatten,,,,,7500,0
7,Bottleneck,Dense-16,Dense,,,,ReLU,16,120016
8,Decoder,Dense-7500,Dense,,,,ReLU,7500,127500
9,Decoder,Reshape,Reshape,,,,,25 x 25 x 12,0


In [9]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_block(ax, x, y, w, h, title, subtitle, color):
    patch = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.025',
                           linewidth=0.9, edgecolor='0.25', facecolor=color)
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h * 0.62, title, ha='center', va='center', fontsize=8, weight='bold')
    ax.text(x + w / 2, y + h * 0.32, subtitle, ha='center', va='center', fontsize=6.8)

def draw_arrow(ax, start, end):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=9, linewidth=0.9, color='0.25'))

fig, ax = plt.subplots(figsize=(7.0, 2.45))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

blocks = [
    (0.02, 0.33, 0.12, 0.34, 'Input', '100 x 150 x 3', '#E8EEF7'),
    (0.18, 0.33, 0.13, 0.34, 'Encoder', 'Conv 4/8/12\nPool 2x2, 2x3', '#D7E8D4'),
    (0.36, 0.33, 0.13, 0.34, 'Latent', 'Dense 16', '#F6E2B8'),
    (0.54, 0.33, 0.13, 0.34, 'Decoder', 'Dense + reshape\nUpsample 2x3, 2x2', '#D7E8D4'),
    (0.72, 0.33, 0.12, 0.34, 'Output', '100 x 150 x 3', '#E8EEF7'),
    (0.88, 0.33, 0.10, 0.34, 'Error', 'x - xhat', '#F1C7C2'),
]
for block in blocks:
    draw_block(ax, *block)
for x0, x1 in [(0.14, 0.18), (0.31, 0.36), (0.49, 0.54), (0.67, 0.72), (0.84, 0.88)]:
    draw_arrow(ax, (x0, 0.50), (x1, 0.50))

ax.text(0.50, 0.88, 'CNN autoencoder with 16-dimensional bottleneck', ha='center', fontsize=9, weight='bold')
ax.text(0.50, 0.12, 'Training uses nominal spectrograms only; anomaly scores are derived from reconstruction error.', ha='center', fontsize=7)
save_figure(fig, 'cnn_ae_architecture')


## CNN-AE Scoring Workflow Figure


In [10]:
fig, ax = plt.subplots(figsize=(7.0, 2.6))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

workflow_blocks = [
    (0.02, 0.52, 0.14, 0.28, 'Spectrogram', 'X/Y/Z RGB\n150 x 100', '#E8EEF7'),
    (0.22, 0.52, 0.15, 0.28, 'CNN-AE', 'reconstructs\ninput image', '#D7E8D4'),
    (0.43, 0.52, 0.15, 0.28, 'Error Map', 'squared / absolute\npixel error', '#F1C7C2'),
    (0.66, 0.66, 0.14, 0.22, 'Global Scores', 'MSE, MAE', '#F6E2B8'),
    (0.66, 0.36, 0.14, 0.22, 'VER Scores', 'max segment\ntop-k segments', '#F6E2B8'),
    (0.86, 0.52, 0.12, 0.28, 'Decision', 'frozen validation\nthreshold', '#E3D7F0'),
]
for block in workflow_blocks:
    draw_block(ax, *block)
draw_arrow(ax, (0.16, 0.66), (0.22, 0.66))
draw_arrow(ax, (0.37, 0.66), (0.43, 0.66))
draw_arrow(ax, (0.58, 0.66), (0.66, 0.77))
draw_arrow(ax, (0.58, 0.62), (0.66, 0.47))
draw_arrow(ax, (0.80, 0.77), (0.86, 0.68))
draw_arrow(ax, (0.80, 0.47), (0.86, 0.60))
ax.text(0.50, 0.93, 'CNN-AE anomaly-scoring workflow', ha='center', fontsize=9, weight='bold')
ax.text(0.50, 0.14, 'Thresholds are selected on validation data and then held fixed for final test evaluation.', ha='center', fontsize=7)
save_figure(fig, 'cnn_ae_scoring_workflow')


## CNN-AE Precision-Recall Curves and Confusion Matrices

These figures are generated when `reports/scores/ae_scores_turning.csv` and `reports/thresholds/ae_thresholds_turning.json` exist. They are produced by notebook 05 after TensorFlow can load the trained CNN-AE model.


In [11]:
AE_SCORE_LABELS = {
    'global_mse': 'Global MSE',
    'global_mae': 'Global MAE',
    'sve_max': 'SVE max',
    'sve_topk': 'SVE top-k',
}

if AE_SCORES_PATH.exists() and AE_THRESHOLDS_PATH.exists():
    ae_scores = pd.read_csv(AE_SCORES_PATH)
    with AE_THRESHOLDS_PATH.open() as f:
        ae_thresholds = json.load(f)
    ae_test = ae_scores[ae_scores['split'] == 'test'].copy()
    ae_test['target'] = (ae_test['label'] == 'chatter').astype(int)

    fig, ax = plt.subplots(figsize=(3.35, 2.65))
    ae_metric_rows = []
    for score_name, label in AE_SCORE_LABELS.items():
        y_true = ae_test['target'].to_numpy()
        score_values = ae_test[score_name].to_numpy()
        recall, precision = interpolated_precision_recall(y_true, score_values)
        ap = average_precision_score(y_true, score_values)
        ax.step(recall, precision, where='post', linewidth=1.8, label=f'{label} ({ap:.3f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.set_title('CNN-AE interpolated precision-recall curves')
    ax.grid(color='0.9', linewidth=0.8)
    ax.legend(frameon=False, loc='lower left')
    save_figure(fig, 'cnn_ae_precision_recall_curves')

    score_names = list(AE_SCORE_LABELS)
    fig, axes = plt.subplots(2, 2, figsize=(5.0, 4.3), constrained_layout=True)
    for ax, score_name in zip(axes.ravel(), score_names):
        threshold = float(ae_thresholds[score_name]['threshold'])
        y_true = ae_test['target'].to_numpy()
        y_pred = (ae_test[score_name].to_numpy() >= threshold).astype(int)
        matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
        ax.imshow(matrix, cmap='Blues', vmin=0)
        for (row, col), value in np.ndenumerate(matrix):
            ax.text(col, row, str(value), ha='center', va='center', color='black', fontsize=9)
        ax.set_xticks([0, 1], labels=['Nominal', 'Chatter'], rotation=35, ha='right')
        ax.set_yticks([0, 1], labels=['Nominal', 'Chatter'])
        ax.set_xlabel('Predicted')
        ax.set_title(AE_SCORE_LABELS[score_name])
        if ax in axes[:, 0]:
            ax.set_ylabel('True')
        tn, fp, fn, tp = matrix.ravel()
        ae_metric_rows.append({'score': score_name, 'threshold': threshold, 'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp})
    save_figure(fig, 'cnn_ae_confusion_matrices')

    ae_confusion_table = pd.DataFrame(ae_metric_rows)
    ae_confusion_table.to_csv(TABLE_DIR / 'cnn_ae_confusion_matrices_publication.csv', index=False)
    ae_confusion_table.to_latex(TABLE_DIR / 'cnn_ae_confusion_matrices_publication.tex', index=False)
    (PUBLICATION_DIR / 'cnn_ae_pr_confusion_pending.md').unlink(missing_ok=True)
    print('Wrote CNN-AE PR curves and confusion matrices.')
else:
    pending_path = PUBLICATION_DIR / 'cnn_ae_pr_confusion_pending.md'
    pending_path.write_text(
        '# CNN-AE PR Curves and Confusion Matrices Pending\n\n'
        'The CNN-AE PR curves and confusion matrices require `reports/scores/ae_scores_turning.csv` '
        'and `reports/thresholds/ae_thresholds_turning.json`. Run notebook 05 after TensorFlow is available '
        'to load the trained `.keras` model and produce those score artifacts.\n'
    )
    print(f'CNN-AE score artifacts not found; wrote {pending_path}')


Wrote CNN-AE PR curves and confusion matrices.


## Artifact Index


In [12]:
artifacts = sorted([p.relative_to(REPO_ROOT).as_posix() for p in PUBLICATION_DIR.rglob('*') if p.is_file()])
artifact_index = pd.DataFrame({'artifact': artifacts})
artifact_index.to_csv(PUBLICATION_DIR / 'artifact_index.csv', index=False)
artifact_index

,artifact
0,reports/publication/artifact_index.csv
1,reports/publication/figures/baseline_confusion...
2,reports/publication/figures/baseline_confusion...
3,reports/publication/figures/baseline_confusion...
4,reports/publication/figures/baseline_metric_pa...
...,...
63,reports/publication/tables/cnn_ae_confusion_ma...
64,reports/publication/tables/turning_method_comp...
65,reports/publication/tables/turning_method_comp...
66,reports/publication/tables/turning_split_count...


## Broach data

In [39]:
# ============================================================
# Broach dataset publication figures and tables
# ============================================================

BROACH_BASELINE_METRICS = (
    REPORT_DIR / "tables" / "metrics_broach_dataset_baselines.csv"
)

BROACH_BASELINE_METRICS_CI = (
    REPORT_DIR / "tables" / "metrics_broach_dataset_baselines_with_ci.csv"
)

BROACH_BASELINE_SCORES = (
    REPORT_DIR / "baselines" / "baseline_scores_broach_dataset.csv"
)

BROACH_BASELINE_THRESHOLDS = (
    REPORT_DIR / "thresholds" / "baseline_thresholds_broach_dataset.json"
)

BROACH_AE_SCORES = (
    REPORT_DIR / "scores" / "ae_scores_broach_dataset.csv"
)

BROACH_AE_THRESHOLDS = (
    REPORT_DIR / "thresholds" / "ae_thresholds_broach_dataset.json"
)

BROACH_AE_METRICS_CI = (
    REPORT_DIR / "tables" / "metrics_broach_dataset_ae_with_ci.csv"
)

# SVE sensitivity study
BROACH_SVE_METRICS = (
    REPORT_DIR / "SVE" / "ae_metrics_sve_sensitivity.csv"
)

BROACH_SVE_SCORES = (
    REPORT_DIR / "SVE" / "sve_scores_broach_dataset.csv"
)

# ------------------------------------------------------------
# Method labels
# ------------------------------------------------------------

BROACH_METHOD_LABELS = {
    "one_class_svm_image_features": "One-class SVM",
    "isolation_forest_image_features": "Isolation Forest",
    "pca_image_reconstruction": "PCA Reconstruction",
}

# ------------------------------------------------------------
# CNN-AE score labels
# ------------------------------------------------------------

BROACH_AE_SCORE_LABELS = {
    "global_mse": "Global MSE",
    "global_mae": "Global MAE",
    "sve_max": "SVE Manual (9 segm.)",
    "sve_topk": "SVE Auto (5 segm.)",
}


### Baseline metrics table 

In [41]:

# ============================================================
# Broach dataset publication table:
# Baselines + best SVE configuration
# ============================================================

baseline_metrics = pd.read_csv(BROACH_BASELINE_METRICS)
ae_metrics = pd.read_csv(REPORT_DIR / "tables" / "metrics_broach_dataset_ae.csv")

manual_row = ae_metrics[ae_metrics["score"] == "sve_max"].iloc[0]

baseline_metrics["Method"] = (
    baseline_metrics["method"]
    .map(BROACH_METHOD_LABELS)
)

# gewünschte Reihenfolge
baseline_order = [
    "One-class SVM",
    "Isolation Forest",
    "PCA Reconstruction",
]

publication_baseline = (
    baseline_metrics[
        [
            "Method",
            "pr_auc",
            "f1",
            "precision",
            "recall",
        ]
    ]
    .rename(
        columns={
            "pr_auc": "PR-AUC",
            "f1": "F1-score",
            "precision": "Precision",
            "recall": "Recall",
        }
    )
)

publication_baseline["Method"] = pd.Categorical(
    publication_baseline["Method"],
    categories=baseline_order,
    ordered=True,
)

publication_baseline = publication_baseline.sort_values(
    "Method"
)

# ------------------------------------------------------------
# Add best SVE configuration from sensitivity study
# ------------------------------------------------------------

sve_metrics = pd.read_csv(BROACH_SVE_METRICS)

best_sve = sve_metrics.loc[
    sve_metrics["pr_auc"].idxmax()
]

publication_baseline = pd.concat(
    [
        publication_baseline,

        pd.DataFrame([
            {
                "Method": BROACH_AE_SCORE_LABELS["sve_max"],
                "PR-AUC": manual_row["pr_auc"],
                "F1-score": manual_row["f1"],
                "Precision": manual_row["precision"],
                "Recall": manual_row["recall"],
            }
        ]),

        pd.DataFrame([
            {
                "Method": BROACH_AE_SCORE_LABELS["sve_topk"],
                "PR-AUC": best_sve["pr_auc"],
                "F1-score": best_sve["f1"],
                "Precision": best_sve["precision"],
                "Recall": best_sve["recall"],
            }
        ])
    ]
)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

publication_baseline.to_csv(
    TABLE_DIR / "broach_baseline_metrics_publication.csv",
    index=False,
)

publication_baseline.to_latex(
    TABLE_DIR / "broach_baseline_metrics_publication.tex",
    index=False,
    float_format="%.3f",
)

publication_baseline

,Method,PR-AUC,F1-score,Precision,Recall
0,One-class SVM,0.917620,0.769231,0.882353,0.681818
1,Isolation Forest,0.759097,0.628571,0.846154,0.500000
2,PCA Reconstruction,0.963292,0.900000,1.000000,0.818182
0,SVE Manual (9 segm.),0.992260,0.926829,1.000000,0.863636
0,SVE Auto (5 segm.),0.891354,0.840000,0.777778,0.913043


### PR Curves for Broach Baselines

In [42]:
# ============================================================
# Broach dataset:
# Baseline precision-recall curves
# ============================================================

baseline_scores = pd.read_csv(BROACH_BASELINE_SCORES)

fig, ax = plt.subplots(figsize=(3.35, 2.65))

method_order = [
    "one_class_svm_image_features",
    "isolation_forest_image_features",
    "pca_image_reconstruction",
]

for method in method_order:

    group = baseline_scores[
        (baseline_scores["split"] == "test")
        & (baseline_scores["method"] == method)
    ]

    y_true = group["target"].to_numpy()
    scores = group["score_value"].to_numpy()

    recall, precision = interpolated_precision_recall(
        y_true,
        scores,
    )

    ap = average_precision_score(
        y_true,
        scores,
    )

    ax.step(
        recall,
        precision,
        where="post",
        linewidth=2.0,
        label=(
            f"{BROACH_METHOD_LABELS.get(method, method)} "
            f"({ap:.3f})"
        ),
    )

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")

ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)

ax.grid(color="0.9")

ax.legend(
    frameon=False,
    fontsize=7,
    ncol=2,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.18),
)

fig.subplots_adjust(top=0.80)

save_figure(
    fig,
    "broach_baseline_precision_recall_curves",
)

### Confusion matrices for Broach dataset baselines

In [43]:
# ============================================================
# Broach dataset:
# Baseline confusion matrices
# ============================================================

with open(BROACH_BASELINE_THRESHOLDS) as f:
    baseline_thresholds = json.load(f)

methods = [
    "one_class_svm_image_features",
    "isolation_forest_image_features",
    "pca_image_reconstruction",
]

fig, axes = plt.subplots(
    1,
    len(methods),
    figsize=(6.8, 2.25),
    constrained_layout=True,
)

confusion_rows = []

for ax, method in zip(axes, methods):

    group = baseline_scores[
        (baseline_scores["split"] == "test")
        & (baseline_scores["method"] == method)
    ]

    threshold = baseline_thresholds[method]["threshold"]

    y_true = group["target"].to_numpy()

    y_pred = (
        group["score_value"].to_numpy()
        >= threshold
    ).astype(int)

    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = matrix.ravel()

    confusion_rows.append(
        {
            "Method": BROACH_METHOD_LABELS[method],
            "Threshold": threshold,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
        }
    )

    ax.imshow(
        matrix,
        cmap="Blues",
        vmin=0,
    )

    for (row, col), value in np.ndenumerate(matrix):

        ax.text(
            col,
            row,
            str(value),
            ha="center",
            va="center",
            fontsize=9,
        )

    ax.set_xticks(
        [0, 1],
        labels=["Normal", "Anomaly"],
        rotation=35,
    )

    ax.set_yticks(
        [0, 1],
        labels=["Normal", "Anomaly"],
    )

    ax.set_xlabel("Predicted")
    ax.set_title(
        BROACH_METHOD_LABELS[method]
    )

axes[0].set_ylabel("True")

save_figure(
    fig,
    "broach_baseline_confusion_matrices",
)

# ------------------------------------------------------------
# Export appendix table
# ------------------------------------------------------------

baseline_confusion_table = pd.DataFrame(
    confusion_rows
)

baseline_confusion_table.to_csv(
    TABLE_DIR / "broach_baseline_confusion_matrices.csv",
    index=False,
)

baseline_confusion_table.to_latex(
    TABLE_DIR / "broach_baseline_confusion_matrices.tex",
    index=False,
)

baseline_confusion_table


,Method,Threshold,TN,FP,FN,TP
0,One-class SVM,15.579819,2476,2,7,15
1,Isolation Forest,0.140067,2476,2,11,11
2,PCA Reconstruction,0.000503,2478,0,4,18


## cnn-ae pr curves and confusion matrices for broach dataset

In [45]:
# ============================================================
# Broach dataset:
# CNN-AE precision-recall curves
# ============================================================

ae_scores = pd.read_csv(BROACH_AE_SCORES)

sve_final_scores = pd.read_csv(
    REPORT_DIR / "SVE" / "sve_final_test_scores.csv"
)

ae_test = ae_scores[
    ae_scores["split"] == "test"
].copy()

y_true = ae_test["label"].astype(int).to_numpy()

fig, ax = plt.subplots(figsize=(3.35, 2.65))

# ------------------------------------------------------------
# Global MSE
# ------------------------------------------------------------

for score_name in [
    "global_mse",
    "global_mae",
    "sve_max",      # Manual SVE (9 segm.)
]:

    label = BROACH_AE_SCORE_LABELS[score_name]

    score_values = ae_test[score_name].to_numpy()

    recall, precision = interpolated_precision_recall(
        y_true,
        score_values,
    )

    pr_auc = average_precision_score(
        y_true,
        score_values,
    )

    ax.step(
        recall,
        precision,
        where="post",
        linewidth=2.0,
        label=f"{label} ({pr_auc:.3f})",
    )

# ------------------------------------------------------------
# Validation-selected SVE
# ------------------------------------------------------------

score_values = sve_final_scores[
    "sve_validation"
].to_numpy()

recall, precision = interpolated_precision_recall(
    y_true,
    score_values,
)

pr_auc = average_precision_score(
    y_true,
    score_values,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2.0,
    linestyle="--",
    label=(
        f"{BROACH_AE_SCORE_LABELS['sve_topk']} "
        f"({pr_auc:.3f})"
    ),
)

# ------------------------------------------------------------
# Plot styling
# ------------------------------------------------------------

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")

ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)

ax.grid(color="0.9")

ax.legend(
    frameon=False,
    fontsize=7,
    ncol=2,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.18),
)

fig.subplots_adjust(top=0.80)

save_figure(
    fig,
    "broach_cnn_ae_precision_recall_curves",
)

In [47]:
# ============================================================
# Broach dataset:
# CNN-AE confusion matrices
# ============================================================

with open(BROACH_AE_THRESHOLDS) as f:
    ae_thresholds = json.load(f)

sve_final_scores = pd.read_csv(
    REPORT_DIR / "SVE" / "sve_final_test_scores.csv"
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(5.0, 4.3),
    constrained_layout=True,
)

confusion_rows = []

# ------------------------------------------------------------
# Global scores + manual SVE
# ------------------------------------------------------------

plot_methods = [
    ("global_mse", BROACH_AE_SCORE_LABELS["global_mse"]),
    ("global_mae", BROACH_AE_SCORE_LABELS["global_mae"]),
    ("sve_max", BROACH_AE_SCORE_LABELS["sve_max"]),
]

for ax, (score_name, label) in zip(
    axes.ravel()[:3],
    plot_methods,
):

    threshold = ae_thresholds[score_name]["threshold"]

    y_pred = (
        ae_test[score_name].to_numpy()
        >= threshold
    ).astype(int)

    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = matrix.ravel()

    confusion_rows.append(
        {
            "Method": label,
            "Threshold": threshold,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
        }
    )

    ax.imshow(
        matrix,
        cmap="Blues",
        vmin=0,
    )

    for (row, col), value in np.ndenumerate(matrix):

        ax.text(
            col,
            row,
            str(value),
            ha="center",
            va="center",
            fontsize=9,
        )

    ax.set_xticks(
        [0, 1],
        labels=["Normal", "Anomaly"],
    )

    ax.set_yticks(
        [0, 1],
        labels=["Normal", "Anomaly"],
    )

    ax.set_title(label)

    ax.set_xlabel("Predicted")

# ------------------------------------------------------------
# Validation-selected SVE
# ------------------------------------------------------------

ax = axes.ravel()[3]

threshold = sve_final_scores[
    "threshold_validation"
].iloc[0]

scores = sve_final_scores[
    "sve_validation"
].to_numpy()

y_pred = (
    scores >= threshold
).astype(int)

matrix = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1],
)

tn, fp, fn, tp = matrix.ravel()

confusion_rows.append(
    {
        "Method": BROACH_AE_SCORE_LABELS["sve_topk"],
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    }
)

ax.imshow(
    matrix,
    cmap="Blues",
    vmin=0,
)

for (row, col), value in np.ndenumerate(matrix):

    ax.text(
        col,
        row,
        str(value),
        ha="center",
        va="center",
        fontsize=9,
    )

ax.set_xticks(
    [0, 1],
    labels=["Normal", "Anomaly"],
)

ax.set_yticks(
    [0, 1],
    labels=["Normal", "Anomaly"],
)

ax.set_title(
    BROACH_AE_SCORE_LABELS["sve_topk"]
)

ax.set_xlabel("Predicted")

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

for ax in axes[:, 0]:
    ax.set_ylabel("True")

save_figure(
    fig,
    "broach_cnn_ae_confusion_matrices",
)

# ------------------------------------------------------------
# Export appendix table
# ------------------------------------------------------------

ae_confusion_table = pd.DataFrame(
    confusion_rows
)

ae_confusion_table.to_csv(
    TABLE_DIR / "broach_ae_confusion_matrices.csv",
    index=False,
)

ae_confusion_table.to_latex(
    TABLE_DIR / "broach_ae_confusion_matrices.tex",
    index=False,
)

ae_confusion_table

,Method,Threshold,TN,FP,FN,TP
0,Global MSE,0.001131,2478,0,11,11
1,Global MAE,0.013949,2477,1,14,8
2,SVE Manual (9 segm.),0.003438,2478,0,3,19
3,SVE Auto (5 segm.),0.001871,2476,2,2,20


In [48]:
# ============================================================
# Broach dataset:
# CNN-AE metrics with confidence intervals
# ============================================================

ae_ci = pd.read_csv(BROACH_AE_METRICS_CI)

selected_scores = list(BROACH_AE_SCORE_LABELS.keys())

publication_ae = ae_ci[
    ae_ci["score"].isin(selected_scores)
].copy()

publication_ae["Method"] = (
    publication_ae["score"]
    .map(BROACH_AE_SCORE_LABELS)
)

# ------------------------------------------------------------
# Format confidence intervals
# ------------------------------------------------------------

publication_ae["PR-AUC [95% CI]"] = publication_ae.apply(
    lambda r:
    f"{r['pr_auc']:.3f} "
    f"[{r['pr_auc_ci_low']:.3f}, "
    f"{r['pr_auc_ci_high']:.3f}]",
    axis=1,
)

publication_ae["F1-score [95% CI]"] = publication_ae.apply(
    lambda r:
    f"{r['f1']:.3f} "
    f"[{r['f1_ci_low']:.3f}, "
    f"{r['f1_ci_high']:.3f}]",
    axis=1,
)

publication_ae["Precision [95% CI]"] = publication_ae.apply(
    lambda r:
    f"{r['precision']:.3f} "
    f"[{r['precision_ci_low']:.3f}, "
    f"{r['precision_ci_high']:.3f}]",
    axis=1,
)

publication_ae["Recall [95% CI]"] = publication_ae.apply(
    lambda r:
    f"{r['recall']:.3f} "
    f"[{r['recall_ci_low']:.3f}, "
    f"{r['recall_ci_high']:.3f}]",
    axis=1,
)

# ------------------------------------------------------------
# Compact publication table
# ------------------------------------------------------------

publication_ae_compact = publication_ae[
    [
        "Method",
        "PR-AUC [95% CI]",
        "F1-score [95% CI]",
        "Precision [95% CI]",
        "Recall [95% CI]",
    ]
]

publication_ae_compact.to_csv(
    TABLE_DIR / "broach_ae_metrics_publication.csv",
    index=False,
)

publication_ae_compact.to_latex(
    TABLE_DIR / "broach_ae_metrics_publication.tex",
    index=False,
)

# ------------------------------------------------------------
# Detailed appendix table
# ------------------------------------------------------------

appendix_ae = publication_ae[
    [
        "Method",
        "pr_auc",
        "f1",
        "precision",
        "recall",
        "tn",
        "fp",
        "fn",
        "tp",
        "pr_auc_ci_low",
        "pr_auc_ci_high",
        "f1_ci_low",
        "f1_ci_high",
        "precision_ci_low",
        "precision_ci_high",
        "recall_ci_low",
        "recall_ci_high",
    ]
].rename(
    columns={
        "pr_auc": "PR-AUC",
        "f1": "F1-score",
        "precision": "Precision",
        "recall": "Recall",
        "tn": "TN",
        "fp": "FP",
        "fn": "FN",
        "tp": "TP",
        "pr_auc_ci_low": "PR-AUC CI Low",
        "pr_auc_ci_high": "PR-AUC CI High",
        "f1_ci_low": "F1 CI Low",
        "f1_ci_high": "F1 CI High",
        "precision_ci_low": "Precision CI Low",
        "precision_ci_high": "Precision CI High",
        "recall_ci_low": "Recall CI Low",
        "recall_ci_high": "Recall CI High",
    }
)

appendix_ae.to_csv(
    TABLE_DIR / "broach_ae_metrics_appendix.csv",
    index=False,
)

appendix_ae.to_latex(
    TABLE_DIR / "broach_ae_metrics_appendix.tex",
    index=False,
    float_format="%.3f",
)

publication_ae_compact


,Method,PR-AUC [95% CI],F1-score [95% CI],Precision [95% CI],Recall [95% CI]
0,Global MSE,"0.593 [0.394, 0.771]","0.667 [0.444, 0.824]","1.000 [1.000, 1.000]","0.500 [0.286, 0.700]"
1,Global MAE,"0.465 [0.274, 0.650]","0.516 [0.273, 0.706]","0.889 [0.636, 1.000]","0.364 [0.167, 0.571]"
2,SVE Manual (9 segm.),"0.992 [0.969, 1.000]","0.927 [0.829, 1.000]","1.000 [1.000, 1.000]","0.864 [0.708, 1.000]"
3,SVE Auto (5 segm.),"0.851 [0.705, 0.959]","0.706 [0.500, 0.857]","1.000 [1.000, 1.000]","0.545 [0.333, 0.750]"


## Create PR-Plot  Baseline vs. AE Result on broaching dataset 

In [51]:
# ============================================================
# Broach dataset:
# Baselines vs. manual and validation-selected SVE
# ============================================================

baseline_scores = pd.read_csv(BROACH_BASELINE_SCORES)

sve_final_scores = pd.read_csv(
    REPORT_DIR / "SVE" / "sve_final_test_scores.csv"
)

fig, ax = plt.subplots(figsize=(3.35, 2.65))

# ------------------------------------------------------------
# Baseline models
# ------------------------------------------------------------

baseline_order = [
    "one_class_svm_image_features",
    "isolation_forest_image_features",
    "pca_image_reconstruction",
]

for method in baseline_order:

    group = baseline_scores[
        (baseline_scores["split"] == "test")
        & (baseline_scores["method"] == method)
    ]

    y_true = group["target"].to_numpy()
    scores = group["score_value"].to_numpy()

    recall, precision = interpolated_precision_recall(
        y_true,
        scores,
    )

    ap = average_precision_score(
        y_true,
        scores,
    )

    ax.step(
        recall,
        precision,
        where="post",
        linewidth=2.0,
        label=(
            f"{BROACH_METHOD_LABELS.get(method, method)} "
            f"({ap:.3f})"
        ),
    )

# ------------------------------------------------------------
# SVE Manual (9 segments)
# ------------------------------------------------------------

y_true_sve = sve_final_scores["label"].to_numpy()

scores = sve_final_scores["sve_manual_9"].to_numpy()

recall, precision = interpolated_precision_recall(
    y_true_sve,
    scores,
)

ap = average_precision_score(
    y_true_sve,
    scores,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2.5,
    linestyle="--",
    label=(
        f"{BROACH_AE_SCORE_LABELS['sve_max']} "
        f"({ap:.3f})"
    ),
)

# ------------------------------------------------------------
# SVE Validation-selected
# ------------------------------------------------------------

scores = sve_final_scores["sve_validation"].to_numpy()

recall, precision = interpolated_precision_recall(
    y_true_sve,
    scores,
)

ap = average_precision_score(
    y_true_sve,
    scores,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2.5,
    linestyle=":",
    label=(
        f"{BROACH_AE_SCORE_LABELS['sve_topk']} "
        f"({ap:.3f})"
    ),
)

# ------------------------------------------------------------
# Plot styling
# ------------------------------------------------------------

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")

ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)

ax.grid(color="0.9")

ax.legend(
    frameon=False,
    fontsize=7,
    ncol=2,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.28),
)

fig.subplots_adjust(top=0.80)

save_figure(
    fig,
    "broach_baseline_vs_sve_pr_curves",
)

### Create Table to summarize baseline vs top AE performance 

In [55]:
# ============================================================
# Broach dataset:
# Global scores vs. manual and validation-selected SVE
# ============================================================

SVE_FINAL_SCORES_PATH = (
    REPORT_DIR / "SVE" / "sve_final_test_scores.csv"
)

alpha = 0.0

sve_scores = pd.read_csv(SVE_FINAL_SCORES_PATH)


def monotonic_precision_recall_curve(y_true, scores):

    precision, recall, _ = precision_recall_curve(
        y_true,
        scores,
    )

    curve = (
        pd.DataFrame(
            {
                "recall": recall,
                "precision": precision,
            }
        )
        .groupby("recall", as_index=False)["precision"]
        .max()
        .sort_values("recall")
    )

    recall = curve["recall"].to_numpy()
    precision = curve["precision"].to_numpy()

    precision = np.maximum.accumulate(
        precision[::-1]
    )[::-1]

    return recall, precision


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(3.35, 2.65))

colors = {
    "global_mse": "#1f77b4",
    "global_mae": "#ff7f0e",
    "sve_max": "#2ca02c",
    "sve_topk": "#d62728",
}

# ------------------------------------------------------------
# Global MSE
# ------------------------------------------------------------

scores = ae_test["global_mse"].to_numpy()

recall, precision = monotonic_precision_recall_curve(
    y_true,
    scores,
)

ap = average_precision_score(
    y_true,
    scores,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2,
    color=colors["global_mse"],
    label=(
        f"{BROACH_AE_SCORE_LABELS['global_mse']} "
        f"({ap:.3f})"
    ),
)

ax.fill_between(
    recall,
    precision,
    step="post",
    alpha=alpha,
    color=colors["global_mse"],
)

# ------------------------------------------------------------
# Global MAE
# ------------------------------------------------------------

scores = ae_test["global_mae"].to_numpy()

recall, precision = monotonic_precision_recall_curve(
    y_true,
    scores,
)

ap = average_precision_score(
    y_true,
    scores,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2,
    color=colors["global_mae"],
    label=(
        f"{BROACH_AE_SCORE_LABELS['global_mae']} "
        f"({ap:.3f})"
    ),
)

ax.fill_between(
    recall,
    precision,
    step="post",
    alpha=alpha,
    color=colors["global_mae"],
)

# ------------------------------------------------------------
# Manual SVE (9 Segments)
# ------------------------------------------------------------

y_true_sve = sve_scores["label"].to_numpy()

scores = sve_scores["sve_manual_9"].to_numpy()

recall, precision = monotonic_precision_recall_curve(
    y_true_sve,
    scores,
)

ap = average_precision_score(
    y_true_sve,
    scores,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2,
    color=colors["sve_max"],
    label=(
        f"{BROACH_AE_SCORE_LABELS['sve_max']} "
        f"({ap:.3f})"
    ),
)

ax.fill_between(
    recall,
    precision,
    step="post",
    alpha=alpha,
    color=colors["sve_max"],
)

# ------------------------------------------------------------
# Validation-selected SVE
# ------------------------------------------------------------

scores = sve_scores["sve_validation"].to_numpy()

recall, precision = monotonic_precision_recall_curve(
    y_true_sve,
    scores,
)

ap = average_precision_score(
    y_true_sve,
    scores,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2,
    color=colors["sve_topk"],
    label=(
        f"{BROACH_AE_SCORE_LABELS['sve_topk']} "
        f"({ap:.3f})"
    ),
)

ax.fill_between(
    recall,
    precision,
    step="post",
    alpha=alpha,
    color=colors["sve_topk"],
)

# ------------------------------------------------------------
# Random baseline
# ------------------------------------------------------------

baseline = y_true.mean()

ax.axhline(
    baseline,
    linestyle="--",
    color="gray",
    linewidth=1,
    label="Random baseline",
)

# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")

ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)

ax.grid(color="0.9")

ax.legend(
    frameon=False,
    fontsize=7,
    ncol=2,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.32),  # etwas höher
)

fig.subplots_adjust(top=0.72)

save_figure(
    fig,
    "broach_global_vs_sve_pr_curves",
)

plt.show()

C:\Users\Zeleny\AppData\Local\Temp\ipykernel_17384\320709644.py:244: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [56]:
# ============================================================
# Broach dataset:
# Global scores vs. manual and validation-selected SVE
# ============================================================

SVE_FINAL_SCORES_PATH = (
    REPORT_DIR / "SVE" / "sve_final_test_scores.csv"
)

alpha = 0.0

sve_scores = pd.read_csv(SVE_FINAL_SCORES_PATH)


def monotonic_precision_recall_curve(y_true, scores):

    precision, recall, _ = precision_recall_curve(
        y_true,
        scores,
    )

    curve = (
        pd.DataFrame(
            {
                "recall": recall,
                "precision": precision,
            }
        )
        .groupby("recall", as_index=False)["precision"]
        .max()
        .sort_values("recall")
    )

    recall = curve["recall"].to_numpy()
    precision = curve["precision"].to_numpy()

    precision = np.maximum.accumulate(
        precision[::-1]
    )[::-1]

    return recall, precision


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(3.35, 2.65))

colors = {
    "Global MSE": "#1f77b4",
    "Global MAE": "#ff7f0e",
    "Manual SVE": "#2ca02c",
    "Validation SVE": "#d62728",
}


# ------------------------------------------------------------
# Global MSE
# ------------------------------------------------------------

scores = ae_test["global_mse"].to_numpy()

recall, precision = monotonic_precision_recall_curve(
    y_true,
    scores,
)

ap = average_precision_score(
    y_true,
    scores,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2,
    color=colors["Global MSE"],
    label=f"Global MSE ({ap:.3f})",
)

ax.fill_between(
    recall,
    precision,
    step="post",
    alpha=alpha,
    color=colors["Global MSE"],
)


# ------------------------------------------------------------
# Global MAE
# ------------------------------------------------------------

scores = ae_test["global_mae"].to_numpy()

recall, precision = monotonic_precision_recall_curve(
    y_true,
    scores,
)

ap = average_precision_score(
    y_true,
    scores,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2,
    color=colors["Global MAE"],
    label=f"Global MAE ({ap:.3f})",
)

ax.fill_between(
    recall,
    precision,
    step="post",
    alpha=alpha,
    color=colors["Global MAE"],
)


# ------------------------------------------------------------
# Manual SVE (9 Segments)
# ------------------------------------------------------------

scores = sve_scores["sve_manual_9"].to_numpy()

recall, precision = monotonic_precision_recall_curve(
    y_true,
    scores,
)

ap = average_precision_score(
    y_true,
    scores,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2,
    color=colors["Manual SVE"],
    label=f"SVE manual (9 seg) ({ap:.3f})",
)

ax.fill_between(
    recall,
    precision,
    step="post",
    alpha=alpha,
    color=colors["Manual SVE"],
)


# ------------------------------------------------------------
# Validation-selected SVE
# ------------------------------------------------------------

scores = sve_scores["sve_validation"].to_numpy()

recall, precision = monotonic_precision_recall_curve(
    y_true,
    scores,
)

ap = average_precision_score(
    y_true,
    scores,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2,
    color=colors["Validation SVE"],
    label=f"SVE validation-selected ({ap:.3f})",
)

ax.fill_between(
    recall,
    precision,
    step="post",
    alpha=alpha,
    color=colors["Validation SVE"],
)


# ------------------------------------------------------------
# Random baseline
# ------------------------------------------------------------

baseline = y_true.mean()

ax.axhline(
    baseline,
    linestyle="--",
    color="gray",
    linewidth=1,
    label="Random baseline",
)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")

ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)

ax.grid(color="0.9")

ax.legend(
    frameon=False,
    ncol=2,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.25),
)

save_figure(
    fig,
    "broach_global_vs_sve_pr_curves",
)

plt.show()

C:\Users\Zeleny\AppData\Local\Temp\ipykernel_17384\3087119570.py:228: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [58]:
# ============================================================
# Broach dataset:
# PR curves for all main anomaly scores
# ============================================================

ae_scores = pd.read_csv(BROACH_AE_SCORES)

sve_final_scores = pd.read_csv(
    REPORT_DIR / "SVE" / "sve_final_test_scores.csv"
)

ae_test = ae_scores[
    ae_scores["split"] == "test"
].copy()

y_true = ae_test["label"].astype(int).to_numpy()

y_true_sve = sve_final_scores["label"].to_numpy()

colors = {
    "global_mse": "#4C78A8",
    "global_mae": "#F58518",
    "sve_max": "#54A24B",
    "sve_topk": "#B279A2",
}

fig, ax = plt.subplots(figsize=(3.35, 2.65))

# ------------------------------------------------------------
# Global MSE
# ------------------------------------------------------------

for score_name in [
    "global_mse",
    "global_mae",
    "sve_max",   # manual configuration
]:

    scores = ae_test[score_name].to_numpy()

    recall, precision = monotonic_precision_recall_curve(
        y_true,
        scores,
    )

    pr_auc = average_precision_score(
        y_true,
        scores,
    )

    ax.step(
        recall,
        precision,
        where="post",
        linewidth=2,
        color=colors[score_name],
        label=(
            f"{BROACH_AE_SCORE_LABELS[score_name]} "
            f"({pr_auc:.3f})"
        ),
    )

# ------------------------------------------------------------
# Validation-selected SVE
# ------------------------------------------------------------

scores = sve_final_scores[
    "sve_validation"
].to_numpy()

recall, precision = monotonic_precision_recall_curve(
    y_true_sve,
    scores,
)

pr_auc = average_precision_score(
    y_true_sve,
    scores,
)

ax.step(
    recall,
    precision,
    where="post",
    linewidth=2,
    color=colors["sve_topk"],
    label=(
        f"{BROACH_AE_SCORE_LABELS['sve_topk']} "
        f"({pr_auc:.3f})"
    ),
)

# ------------------------------------------------------------
# Random baseline
# ------------------------------------------------------------

baseline = y_true.mean()

ax.axhline(
    baseline,
    linestyle="--",
    color="gray",
    linewidth=1,
    label=f"Random baseline ({baseline:.3f})",
)

# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")

ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)

ax.grid(color="0.9")

ax.legend(
    frameon=False,
    fontsize=7,
    ncol=2,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.30),
)

fig.subplots_adjust(top=0.72)

save_figure(
    fig,
    "broach_ae_score_comparison_pr_curves",
)

In [59]:

# ============================================================
# Appendix table:
# Detailed metrics with confidence intervals
# ============================================================

baseline_ci = pd.read_csv(
    BROACH_BASELINE_METRICS_CI
)

ae_ci = pd.read_csv(
    BROACH_AE_METRICS_CI
)

rows = []

# ------------------------------------------------------------
# Baseline methods
# ------------------------------------------------------------

for _, row in baseline_ci.iterrows():

    rows.append({
        "Method": BROACH_METHOD_LABELS.get(
            row["method"],
            row["method"],
        ),

        "PR-AUC": row["pr_auc"],
        "F1-score": row["f1"],
        "Precision": row["precision"],
        "Recall": row["recall"],

        "TN": row["tn"],
        "FP": row["fp"],
        "FN": row["fn"],
        "TP": row["tp"],

        "PR-AUC 95% CI":
            f"[{row['pr_auc_ci_low']:.3f}, "
            f"{row['pr_auc_ci_high']:.3f}]",

        "F1-score 95% CI":
            f"[{row['f1_ci_low']:.3f}, "
            f"{row['f1_ci_high']:.3f}]",

        "Precision 95% CI":
            f"[{row['precision_ci_low']:.3f}, "
            f"{row['precision_ci_high']:.3f}]",

        "Recall 95% CI":
            f"[{row['recall_ci_low']:.3f}, "
            f"{row['recall_ci_high']:.3f}]",
    })

# ------------------------------------------------------------
# CNN-AE methods
# ------------------------------------------------------------

for _, row in ae_ci.iterrows():

    if row["score"] not in BROACH_AE_SCORE_LABELS:
        continue

    rows.append({
        "Method": BROACH_AE_SCORE_LABELS[
            row["score"]
        ],

        "PR-AUC": row["pr_auc"],
        "F1-score": row["f1"],
        "Precision": row["precision"],
        "Recall": row["recall"],

        "TN": row["tn"],
        "FP": row["fp"],
        "FN": row["fn"],
        "TP": row["tp"],

        "PR-AUC 95% CI":
            f"[{row['pr_auc_ci_low']:.3f}, "
            f"{row['pr_auc_ci_high']:.3f}]",

        "F1-score 95% CI":
            f"[{row['f1_ci_low']:.3f}, "
            f"{row['f1_ci_high']:.3f}]",

        "Precision 95% CI":
            f"[{row['precision_ci_low']:.3f}, "
            f"{row['precision_ci_high']:.3f}]",

        "Recall 95% CI":
            f"[{row['recall_ci_low']:.3f}, "
            f"{row['recall_ci_high']:.3f}]",
    })

appendix_ci_table = pd.DataFrame(rows)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

appendix_ci_table.to_csv(
    TABLE_DIR / "broach_appendix_ci_metrics.csv",
    index=False,
)

appendix_ci_table.to_latex(
    TABLE_DIR / "broach_appendix_ci_metrics.tex",
    index=False,
    float_format="%.3f",
)

appendix_ci_table


,Method,PR-AUC,F1-score,Precision,Recall,TN,FP,FN,TP,PR-AUC 95% CI,F1-score 95% CI,Precision 95% CI,Recall 95% CI
0,Isolation Forest,0.759097,0.628571,0.846154,0.500000,2476,2,11,11,"[0.581, 0.899]","[0.421, 0.800]","[0.615, 1.000]","[0.300, 0.720]"
1,One-class SVM,0.917620,0.769231,0.882353,0.681818,2476,2,7,15,"[0.814, 0.989]","[0.588, 0.898]","[0.706, 1.000]","[0.470, 0.864]"
2,PCA Reconstruction,0.963292,0.900000,1.000000,0.818182,2478,0,4,18,"[0.894, 1.000]","[0.788, 0.980]","[1.000, 1.000]","[0.650, 0.960]"
3,Global MSE,0.592931,0.666667,1.000000,0.500000,2478,0,11,11,"[0.394, 0.771]","[0.444, 0.824]","[1.000, 1.000]","[0.286, 0.700]"
4,Global MAE,0.465159,0.516129,0.888889,0.363636,2477,1,14,8,"[0.274, 0.650]","[0.273, 0.706]","[0.636, 1.000]","[0.167, 0.571]"
5,SVE Manual (9 segm.),0.992260,0.926829,1.000000,0.863636,2478,0,3,19,"[0.969, 1.000]","[0.829, 1.000]","[1.000, 1.000]","[0.708, 1.000]"
6,SVE Auto (5 segm.),0.850703,0.705882,1.000000,0.545455,2478,0,10,12,"[0.705, 0.959]","[0.500, 0.857]","[1.000, 1.000]","[0.333, 0.750]"
